In [12]:
import pandas as pd
import plotly.express as px
import json
import os

# 1. Load raw listings
listings = pd.read_csv('../data/raw/bangkok/listings.csv.gz', compression='gzip', low_memory=False)

# 2. Clean price, flag and remove deterrent-pricing outliers
listings['price_clean'] = (
    listings['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)
listings['price_flag_suspicious'] = (listings['price_clean'] > 50000) | (listings['price_clean'] < 100)
listings_clean = listings[~listings['price_flag_suspicious']].copy()
print(f"Listings loaded and cleaned: {listings_clean.shape}")

# 3. Aggregate per neighbourhood
neigh_stats = (
    listings_clean
    .groupby('neighbourhood_cleansed')
    .agg(
        listing_count=('id', 'count'),
        median_price=('price_clean', 'median'),
        avg_reviews=('number_of_reviews', 'mean'),
        avg_availability=('availability_365', 'mean'),
    )
    .reset_index()
)

# 4. Load geojson
with open('../data/raw/bangkok/neighbourhoods.geojson', encoding='utf-8') as f:
    geojson_data = json.load(f)

# 5. Build all three figures
fig_map = px.choropleth_mapbox(
    neigh_stats, geojson=geojson_data, locations='neighbourhood_cleansed',
    featureidkey='properties.neighbourhood', color='median_price',
    color_continuous_scale='YlOrRd', mapbox_style='carto-positron',
    center={'lat': 13.7563, 'lon': 100.5018}, zoom=9, opacity=0.7,
    hover_data=['listing_count', 'avg_reviews'],
    title='Median Listing Price by Neighbourhood — Bangkok'
)
fig_map.update_layout(margin={'r': 0, 't': 40, 'l': 0, 'b': 0})

top_neigh = neigh_stats.sort_values('listing_count', ascending=False).head(15)
fig_bar = px.bar(
    top_neigh, x='listing_count', y='neighbourhood_cleansed', orientation='h',
    title='Top 15 Neighbourhoods by Listing Count — Bangkok',
    labels={'listing_count': 'Number of Listings', 'neighbourhood_cleansed': 'Neighbourhood'}
)
fig_bar.update_layout(yaxis={'categoryorder': 'total ascending'})

fig_box = px.box(
    listings_clean, x='room_type', y='price_clean',
    title='Price Distribution by Room Type — Bangkok',
    labels={'price_clean': 'Price (THB)', 'room_type': 'Room Type'}
)
fig_box.update_yaxes(range=[0, 8000])

print("All three figures built: fig_map, fig_bar, fig_box")

Listings loaded and cleaned: (28766, 81)


C:\Users\Alokaperera\AppData\Local\Temp\ipykernel_22076\3454252100.py:38: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_map = px.choropleth_mapbox(


All three figures built: fig_map, fig_bar, fig_box


In [15]:
import os

os.makedirs('../reports', exist_ok=True)

dashboard_html = f"""
<html>
<head>
    <title>Bangkok Airbnb Market Explorer</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; background: #fafafa; }}
        h1 {{ color: #2c3e50; }}
        .chart {{ margin-bottom: 50px; }}S
    </style>
</head>
<body>
    <h1>Bangkok Airbnb Market Explorer</h1>
    <p>Interactive analysis of Bangkok Airbnb listings - pricing, supply concentration, and room type breakdown.</p>

    <div class="chart">{fig_map.to_html(full_html=False, include_plotlyjs='cdn')}</div>
    <div class="chart">{fig_bar.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="chart">{fig_box.to_html(full_html=False, include_plotlyjs=False)}</div>
</body>
</html>
"""

with open('../reports/bangkok_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(dashboard_html)

print(os.path.abspath('../reports/bangkok_dashboard.html'))

C:\Users\Alokaperera\Downloads\airbnb_project\reports\bangkok_dashboard.html
